In [1]:
import spark
from pyspark.sql import SparkSession

### Explanation:
- `SparkSession`: This is the entry point to use PySpark. It allows us to create DataFrames and interact with Spark.
- `builder`: Used to configure the Spark session.
- `appName`: Sets a name for the Spark application.
- `getOrCreate()`: Creates a new Spark session or reuses an existing one.

In [2]:
spark = SparkSession.builder \
        .appName("Handling_DataTypes") \
        .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/11 17:30:22 INFO SparkEnv: Registering MapOutputTracker
26/04/11 17:30:22 INFO SparkEnv: Registering BlockManagerMaster
26/04/11 17:30:22 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/04/11 17:30:22 INFO SparkEnv: Registering OutputCommitCoordinator


#### Sample data

In [5]:

data = [
    (1, "John Doe", "Bangalore", "2023-01-15", "152.75", "True"),
    (2, "Jane Smith", "Delhi", "2023-05-20", "89.50", "False"),
    (3, "Robert Brown", "Mumbai", "InvalidDate", "200.00", "True"),
    (4, "Linda White", "Kolkata", "2023-02-29", None, "yes"),  # Feb 29 invalid in 2023
    (5, "Mike Green", "Chennai", "2023-08-10", "NaN", "1"),  # NaN needs handling
    (6, "Sarah Blue", "Hyderabad", "InvalidDate", "300.40", "No")
]


#### Define Column names

In [7]:
columns = ['id', 'name', 'city', 'date', 'amount', 'is_active']

#### Create DataFrame: A DataFrame is a distributed collection of data organized with columns.

In [8]:
df = spark.createDataFrame(data, columns)

In [9]:
# Show the DataFrame
df.show()

+---+------------+---------+-----------+------+---------+
| id|        name|     city|       date|amount|is_active|
+---+------------+---------+-----------+------+---------+
|  1|    John Doe|Bangalore| 2023-01-15|152.75|     True|
|  2|  Jane Smith|    Delhi| 2023-05-20| 89.50|    False|
|  3|Robert Brown|   Mumbai|InvalidDate|200.00|     True|
|  4| Linda White|  Kolkata| 2023-02-29|  NULL|      yes|
|  5|  Mike Green|  Chennai| 2023-08-10|   NaN|        1|
|  6|  Sarah Blue|Hyderabad|InvalidDate|300.40|       No|
+---+------------+---------+-----------+------+---------+



In [10]:
# Check the schema
df.printSchema()

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- date: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- is_active: string (nullable = true)



#### Handling Integer Column

In [12]:
df['id']

Column<'id'>

In [11]:
df.select('id').show()

+---+
| id|
+---+
|  1|
|  2|
|  3|
|  4|
|  5|
|  6|
+---+



In [15]:
df.filter(df.id > 3).show()

+---+-----------+---------+-----------+------+---------+
| id|       name|     city|       date|amount|is_active|
+---+-----------+---------+-----------+------+---------+
|  4|Linda White|  Kolkata| 2023-02-29|  NULL|      yes|
|  5| Mike Green|  Chennai| 2023-08-10|   NaN|        1|
|  6| Sarah Blue|Hyderabad|InvalidDate|300.40|       No|
+---+-----------+---------+-----------+------+---------+



In [16]:
df.withColumn("id_double", df.id*2).show()

+---+------------+---------+-----------+------+---------+---------+
| id|        name|     city|       date|amount|is_active|id_double|
+---+------------+---------+-----------+------+---------+---------+
|  1|    John Doe|Bangalore| 2023-01-15|152.75|     True|        2|
|  2|  Jane Smith|    Delhi| 2023-05-20| 89.50|    False|        4|
|  3|Robert Brown|   Mumbai|InvalidDate|200.00|     True|        6|
|  4| Linda White|  Kolkata| 2023-02-29|  NULL|      yes|        8|
|  5|  Mike Green|  Chennai| 2023-08-10|   NaN|        1|       10|
|  6|  Sarah Blue|Hyderabad|InvalidDate|300.40|       No|       12|
+---+------------+---------+-----------+------+---------+---------+



#### Casting the id column to IntegerType

In [17]:
from pyspark.sql.types import IntegerType
from pyspark.sql.functions import col

df = df.withColumn('id', col('id').cast(IntegerType()))

In [18]:
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- date: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- is_active: string (nullable = true)



#### Handling String Columns

In [19]:
from pyspark.sql.functions import *

df = df.withColumn('name_upper', upper(df.name))
df.show()

+---+------------+---------+-----------+------+---------+------------+
| id|        name|     city|       date|amount|is_active|  name_upper|
+---+------------+---------+-----------+------+---------+------------+
|  1|    John Doe|Bangalore| 2023-01-15|152.75|     True|    JOHN DOE|
|  2|  Jane Smith|    Delhi| 2023-05-20| 89.50|    False|  JANE SMITH|
|  3|Robert Brown|   Mumbai|InvalidDate|200.00|     True|ROBERT BROWN|
|  4| Linda White|  Kolkata| 2023-02-29|  NULL|      yes| LINDA WHITE|
|  5|  Mike Green|  Chennai| 2023-08-10|   NaN|        1|  MIKE GREEN|
|  6|  Sarah Blue|Hyderabad|InvalidDate|300.40|       No|  SARAH BLUE|
+---+------------+---------+-----------+------+---------+------------+



In [20]:
df.filter(df.city.startswith('B')).show()

+---+--------+---------+----------+------+---------+----------+
| id|    name|     city|      date|amount|is_active|name_upper|
+---+--------+---------+----------+------+---------+----------+
|  1|John Doe|Bangalore|2023-01-15|152.75|     True|  JOHN DOE|
+---+--------+---------+----------+------+---------+----------+



In [21]:
df = df.withColumn('name_lower', lower(df.name))
df.show()

+---+------------+---------+-----------+------+---------+------------+------------+
| id|        name|     city|       date|amount|is_active|  name_upper|  name_lower|
+---+------------+---------+-----------+------+---------+------------+------------+
|  1|    John Doe|Bangalore| 2023-01-15|152.75|     True|    JOHN DOE|    john doe|
|  2|  Jane Smith|    Delhi| 2023-05-20| 89.50|    False|  JANE SMITH|  jane smith|
|  3|Robert Brown|   Mumbai|InvalidDate|200.00|     True|ROBERT BROWN|robert brown|
|  4| Linda White|  Kolkata| 2023-02-29|  NULL|      yes| LINDA WHITE| linda white|
|  5|  Mike Green|  Chennai| 2023-08-10|   NaN|        1|  MIKE GREEN|  mike green|
|  6|  Sarah Blue|Hyderabad|InvalidDate|300.40|       No|  SARAH BLUE|  sarah blue|
+---+------------+---------+-----------+------+---------+------------+------------+



#### Handle Float Column

In [22]:
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- date: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- is_active: string (nullable = true)
 |-- name_upper: string (nullable = true)
 |-- name_lower: string (nullable = true)



In [23]:
df.show()

+---+------------+---------+-----------+------+---------+------------+------------+
| id|        name|     city|       date|amount|is_active|  name_upper|  name_lower|
+---+------------+---------+-----------+------+---------+------------+------------+
|  1|    John Doe|Bangalore| 2023-01-15|152.75|     True|    JOHN DOE|    john doe|
|  2|  Jane Smith|    Delhi| 2023-05-20| 89.50|    False|  JANE SMITH|  jane smith|
|  3|Robert Brown|   Mumbai|InvalidDate|200.00|     True|ROBERT BROWN|robert brown|
|  4| Linda White|  Kolkata| 2023-02-29|  NULL|      yes| LINDA WHITE| linda white|
|  5|  Mike Green|  Chennai| 2023-08-10|   NaN|        1|  MIKE GREEN|  mike green|
|  6|  Sarah Blue|Hyderabad|InvalidDate|300.40|       No|  SARAH BLUE|  sarah blue|
+---+------------+---------+-----------+------+---------+------------+------------+



In [27]:
df = df.withColumn('amount', col('amount').cast('float'))

In [28]:
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- date: string (nullable = true)
 |-- amount: float (nullable = true)
 |-- is_active: string (nullable = true)
 |-- name_upper: string (nullable = true)
 |-- name_lower: string (nullable = true)



In [29]:
df_filled = df.fillna({'amount': 0})
df_filled.show()

+---+------------+---------+-----------+------+---------+------------+------------+
| id|        name|     city|       date|amount|is_active|  name_upper|  name_lower|
+---+------------+---------+-----------+------+---------+------------+------------+
|  1|    John Doe|Bangalore| 2023-01-15|152.75|     True|    JOHN DOE|    john doe|
|  2|  Jane Smith|    Delhi| 2023-05-20|  89.5|    False|  JANE SMITH|  jane smith|
|  3|Robert Brown|   Mumbai|InvalidDate| 200.0|     True|ROBERT BROWN|robert brown|
|  4| Linda White|  Kolkata| 2023-02-29|   0.0|      yes| LINDA WHITE| linda white|
|  5|  Mike Green|  Chennai| 2023-08-10|   0.0|        1|  MIKE GREEN|  mike green|
|  6|  Sarah Blue|Hyderabad|InvalidDate| 300.4|       No|  SARAH BLUE|  sarah blue|
+---+------------+---------+-----------+------+---------+------------+------------+



### Explanation:
- `filter()`: Filters rows based on a condition.
- `withColumn()`: Adds a new column/replace existing column. We add a new column `id_double`.
- `upper()`: Converts a string column to uppercase.
- `startswith()`: Filters rows where the string column starts with a specific value.
- `cast()`: Converts a column to a specific data type. above, we convert `amount` to `float`.
- `selectExpr()`: Allows us to run SQL expressions. Here, we calculate the average of the `amount` column.
- `to_date()`: Converts a string column to a date column using the specified format.
- `isNotNull()`: Filters rows where the column is not null.
- `when()`: Used for conditional logic. Here, we convert `is_active` to a boolean column.
- `lit()`: Adds a constant value to a column.
- `amount + 10`: Performs arithmetic operations on a column.

#### Handling Date Column: Using different data

In [32]:
# Sending the Data To HDFS Via Notebook

# Create a CSV file
csv_data = """id,date_iso,date_dmy,date_mdy,timestamp
1,2023-01-15,15/01/2023,01/15/2023,2023-01-15 10:30:00
2,2023-05-20,20/05/2023,05/20/2023,2023-05-20 15:45:00
3,InvalidDate,31/02/2023,02/31/2023,InvalidTimestamp
4,,,,
"""

# Save the CSV file
with open("dates_data.csv", "w") as f:
    f.write(csv_data)


In [33]:
!ls *dates_data*

dates_data.csv


In [3]:
!hadoop fs -put dates_data.csv /data/dates_data.csv

^C


In [4]:
!hadoop fs -ls /data/

Found 7 items
-rw-r--r--   2 root hadoop    1060750 2026-04-10 21:42 /data/customers.csv
-rw-r--r--   2 root hadoop       5488 2026-04-10 15:25 /data/customers_100.csv
-rw-r--r--   2 root hadoop   10528211 2026-04-10 21:41 /data/customers_10mb.csv
-rw-r--r--   2 root hadoop  168541068 2026-04-10 21:40 /data/customers_150mb.csv
-rw-r--r--   2 root hadoop  236978176 2026-04-10 16:31 /data/customers_300mb.csv
-rw-r--r--   2 root hadoop        209 2026-04-11 17:27 /data/dates_data.csv
drwxr-xr-x   - root hadoop          0 2026-04-11 15:48 /data/write_output.csv


In [11]:
## Used for checking the column datype as date, string, and timestamp and looking how does that affect the ouput data
# DDL String for the schema
ddl_schema = """
    id INT,
    date_iso DATE,
    date_dmy DATE,
    date_mdy DATE,
    timestamp TIMESTAMP
"""


# Read the CSV file into a DataFrame
df_file = spark.read.option("header", True).schema(ddl_schema).csv("/data/dates_data.csv")



# Show the DataFrame
df_file.show(truncate=False)

+---+----------+--------+--------+-------------------+
|id |date_iso  |date_dmy|date_mdy|timestamp          |
+---+----------+--------+--------+-------------------+
|1  |2023-01-15|NULL    |NULL    |2023-01-15 10:30:00|
|2  |2023-05-20|NULL    |NULL    |2023-05-20 15:45:00|
|3  |NULL      |NULL    |NULL    |NULL               |
|4  |NULL      |NULL    |NULL    |NULL               |
+---+----------+--------+--------+-------------------+



In [12]:
df_file.printSchema()

root
 |-- id: integer (nullable = true)
 |-- date_iso: date (nullable = true)
 |-- date_dmy: date (nullable = true)
 |-- date_mdy: date (nullable = true)
 |-- timestamp: timestamp (nullable = true)



In [13]:
# Sample data with multiple date formats
date_data = [
    (1, "2023-01-15", "15/01/2023", "01/15/2023", "2023-01-15 10:30:00"),
    (2, "2023-05-20", "20/05/2023", "05/20/2023", "2023-05-20 15:45:00"),
    (3, "InvalidDate", "31/02/2023", "02/31/2023", "InvalidTimestamp"),  # Invalid dates
    (4, None, None, None, None)  # Null values
]

# Define column names
columns = ["id", "date_iso", "date_dmy", "date_mdy", "timestamp"]


# Create DataFrame
df = spark.createDataFrame(date_data, schema=columns)

# Show the DataFrame
df.show(truncate=False)

+---+-----------+----------+----------+-------------------+
|id |date_iso   |date_dmy  |date_mdy  |timestamp          |
+---+-----------+----------+----------+-------------------+
|1  |2023-01-15 |15/01/2023|01/15/2023|2023-01-15 10:30:00|
|2  |2023-05-20 |20/05/2023|05/20/2023|2023-05-20 15:45:00|
|3  |InvalidDate|31/02/2023|02/31/2023|InvalidTimestamp   |
|4  |NULL       |NULL      |NULL      |NULL               |
+---+-----------+----------+----------+-------------------+



In [14]:
df.printSchema()

root
 |-- id: long (nullable = true)
 |-- date_iso: string (nullable = true)
 |-- date_dmy: string (nullable = true)
 |-- date_mdy: string (nullable = true)
 |-- timestamp: string (nullable = true)



In [18]:
from pyspark.sql.functions import to_date

df = df.withColumn('parsed_date_iso', to_date(df.date_iso , 'yyyy-MM-dd')) \
       .withColumn('parsed_date_dmy', to_date(df.date_dmy , 'dd/MM/yyyy')) \
       .withColumn('parsed_date_mdy', to_date(df.date_mdy , 'MM/dd/yyyy'))

In [19]:
df.show(truncate = False)

+---+-----------+----------+----------+-------------------+---------------+---------------+---------------+
|id |date_iso   |date_dmy  |date_mdy  |timestamp          |parsed_date_iso|parsed_date_dmy|parsed_date_mdy|
+---+-----------+----------+----------+-------------------+---------------+---------------+---------------+
|1  |2023-01-15 |15/01/2023|01/15/2023|2023-01-15 10:30:00|2023-01-15     |2023-01-15     |2023-01-15     |
|2  |2023-05-20 |20/05/2023|05/20/2023|2023-05-20 15:45:00|2023-05-20     |2023-05-20     |2023-05-20     |
|3  |InvalidDate|31/02/2023|02/31/2023|InvalidTimestamp   |NULL           |NULL           |NULL           |
|4  |NULL       |NULL      |NULL      |NULL               |NULL           |NULL           |NULL           |
+---+-----------+----------+----------+-------------------+---------------+---------------+---------------+



#### TimeStamp

In [20]:
from pyspark.sql.functions import to_timestamp, year, month, dayofmonth, hour, minute

In [23]:
df = df.withColumn('parsed_timestamp', to_timestamp(df.timestamp, 'yyyy-MM-dd HH:mm:ss'))
df.show()

+---+-----------+----------+----------+-------------------+---------------+---------------+---------------+-------------------+
| id|   date_iso|  date_dmy|  date_mdy|          timestamp|parsed_date_iso|parsed_date_dmy|parsed_date_mdy|   parsed_timestamp|
+---+-----------+----------+----------+-------------------+---------------+---------------+---------------+-------------------+
|  1| 2023-01-15|15/01/2023|01/15/2023|2023-01-15 10:30:00|     2023-01-15|     2023-01-15|     2023-01-15|2023-01-15 10:30:00|
|  2| 2023-05-20|20/05/2023|05/20/2023|2023-05-20 15:45:00|     2023-05-20|     2023-05-20|     2023-05-20|2023-05-20 15:45:00|
|  3|InvalidDate|31/02/2023|02/31/2023|   InvalidTimestamp|           NULL|           NULL|           NULL|               NULL|
|  4|       NULL|      NULL|      NULL|               NULL|           NULL|           NULL|           NULL|               NULL|
+---+-----------+----------+----------+-------------------+---------------+---------------+-------------

In [24]:
df.printSchema()

root
 |-- id: long (nullable = true)
 |-- date_iso: string (nullable = true)
 |-- date_dmy: string (nullable = true)
 |-- date_mdy: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- parsed_date_iso: date (nullable = true)
 |-- parsed_date_dmy: date (nullable = true)
 |-- parsed_date_mdy: date (nullable = true)
 |-- parsed_timestamp: timestamp (nullable = true)



#### From the parsed timestamp, I am getting the year,month,day,hour,minute

In [25]:
df = df.withColumn('year', year(df.parsed_timestamp))\
       .withColumn('month', month(df.parsed_timestamp))\
       .withColumn('day', dayofmonth(df.parsed_timestamp))\
       .withColumn('hour', hour(df.parsed_timestamp))\
       .withColumn('minute', minute(df.parsed_timestamp))

In [26]:
df.show()

+---+-----------+----------+----------+-------------------+---------------+---------------+---------------+-------------------+----+-----+----+----+------+
| id|   date_iso|  date_dmy|  date_mdy|          timestamp|parsed_date_iso|parsed_date_dmy|parsed_date_mdy|   parsed_timestamp|year|month| day|hour|minute|
+---+-----------+----------+----------+-------------------+---------------+---------------+---------------+-------------------+----+-----+----+----+------+
|  1| 2023-01-15|15/01/2023|01/15/2023|2023-01-15 10:30:00|     2023-01-15|     2023-01-15|     2023-01-15|2023-01-15 10:30:00|2023|    1|  15|  10|    30|
|  2| 2023-05-20|20/05/2023|05/20/2023|2023-05-20 15:45:00|     2023-05-20|     2023-05-20|     2023-05-20|2023-05-20 15:45:00|2023|    5|  20|  15|    45|
|  3|InvalidDate|31/02/2023|02/31/2023|   InvalidTimestamp|           NULL|           NULL|           NULL|               NULL|NULL| NULL|NULL|NULL|  NULL|
|  4|       NULL|      NULL|      NULL|               NULL|     

#### Getting the date difference

In [27]:
from pyspark.sql.functions import datediff

## datediff(end_date, start_date) ----> parsed_date_mdy - parsed_date_iso

df = df.withColumn('days_difference' , datediff(df.parsed_date_mdy, df.parsed_date_iso))

In [28]:
df.show()

+---+-----------+----------+----------+-------------------+---------------+---------------+---------------+-------------------+----+-----+----+----+------+---------------+
| id|   date_iso|  date_dmy|  date_mdy|          timestamp|parsed_date_iso|parsed_date_dmy|parsed_date_mdy|   parsed_timestamp|year|month| day|hour|minute|days_difference|
+---+-----------+----------+----------+-------------------+---------------+---------------+---------------+-------------------+----+-----+----+----+------+---------------+
|  1| 2023-01-15|15/01/2023|01/15/2023|2023-01-15 10:30:00|     2023-01-15|     2023-01-15|     2023-01-15|2023-01-15 10:30:00|2023|    1|  15|  10|    30|              0|
|  2| 2023-05-20|20/05/2023|05/20/2023|2023-05-20 15:45:00|     2023-05-20|     2023-05-20|     2023-05-20|2023-05-20 15:45:00|2023|    5|  20|  15|    45|              0|
|  3|InvalidDate|31/02/2023|02/31/2023|   InvalidTimestamp|           NULL|           NULL|           NULL|               NULL|NULL| NULL|NU

In [29]:
df.select('parsed_date_mdy','parsed_date_iso','days_difference').show(truncate=False)

+---------------+---------------+---------------+
|parsed_date_mdy|parsed_date_iso|days_difference|
+---------------+---------------+---------------+
|2023-01-15     |2023-01-15     |0              |
|2023-05-20     |2023-05-20     |0              |
|NULL           |NULL           |NULL           |
|NULL           |NULL           |NULL           |
+---------------+---------------+---------------+



In [30]:
spark.stop()